# MedGemma T1 QLoRA

This notebook runs one class-balanced public T1 microscopy retraining experiment only. Synthetic T2 records are excluded because they do not contain images. Training uses only the training split; validation is evaluated separately for model selection, and the test split is not loaded by this notebook.

In [ ]:
!nvidia-smi

%pip install -q --upgrade \
    "transformers>=4.50.0" \
    "peft>=0.12.0" \
    "bitsandbytes>=0.43.1" \
    "accelerate>=0.33.0" \
    "scikit-learn>=1.4.0"

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
assert os.environ["HF_TOKEN"]
print("HF_TOKEN loaded")

In [ ]:
from pathlib import Path
import importlib.metadata as metadata
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
assert torch.cuda.is_available(), "A CUDA GPU is required"

input_root = Path("/kaggle/input")
config_matches = list(input_root.rglob("configs/secondary.json"))
if not config_matches:
    raise FileNotFoundError("Attach the secondary-data dataset in Kaggle's Add Data panel")
ROOT = config_matches[0].parent.parent
T1_SCRIPT = ROOT / "scripts/train_t1_qlora.py"
if not T1_SCRIPT.exists():
    raise FileNotFoundError("Attach the latest medgemma-secondary-kaggle dataset version containing scripts/train_t1_qlora.py")
if "--class-balanced" not in T1_SCRIPT.read_text():
    raise RuntimeError("Attached medgemma-secondary-kaggle version is stale; publish/attach the version containing the class-balanced trainer.")

required = [
    ROOT / "configs/secondary.json",
    ROOT / "data/secondary/instruction_tuning_mixed/train.jsonl",
    ROOT / "data/secondary/instruction_tuning_mixed/val.jsonl",
    ROOT / "data/secondary/processed/microscopy/tryp",
]
assert all(path.exists() for path in required)
print("Using dataset root:", ROOT)
print("T1 runner:", T1_SCRIPT)
print("T1 train records:", sum(1 for line in (ROOT / "data/secondary/instruction_tuning_mixed/train.jsonl").read_text().splitlines() if line.strip()))
print("T1 validation records:", sum(1 for line in (ROOT / "data/secondary/instruction_tuning_mixed/val.jsonl").read_text().splitlines() if line.strip()))

In [ ]:
import subprocess
import sys

OUTPUT_DIR = Path("/kaggle/working/medgemma-t1-qlora-balanced")
adapter_path = OUTPUT_DIR / "adapter_model.safetensors"
# Class balancing cycles the 65 negative records to match 2,166 positives (4,332 effective examples).
TRAIN_LIMIT = None

if adapter_path.exists():
    print("Existing adapter found; skipping training:", adapter_path)
else:
    command = [
        sys.executable, "-u", str(T1_SCRIPT),
        "--root", str(ROOT),
        "--output", str(OUTPUT_DIR),
        "--epochs", "1",
        "--learning-rate", "2e-4",
        "--gradient-accumulation-steps", "8",
        "--seed", "42",
        "--class-balanced",
        "--skip-eval",
    ]
    if TRAIN_LIMIT is not None:
        command.extend(["--max-train", str(TRAIN_LIMIT)])
    subprocess.run(command, check=True)

In [ ]:
assert adapter_path.exists(), f"Adapter was not saved: {adapter_path}"
print("Adapter saved:", adapter_path)
print("Adapter size:", adapter_path.stat().st_size)
print("Output files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" -", path.name)

In [ ]:
metrics_path = OUTPUT_DIR / "t1_metrics.json"
if metrics_path.exists():
    print("Existing metrics found; skipping validation:", metrics_path)
else:
    command = [
        sys.executable, "-u", str(T1_SCRIPT),
        "--root", str(ROOT),
        "--output", str(OUTPUT_DIR),
        "--eval-only",
        "--max-val", "488",
        "--max-new-tokens", "8",
    ]
    subprocess.run(command, check=True)

In [ ]:
import json

assert metrics_path.exists(), f"Metrics were not written: {metrics_path}"
metrics = json.loads(metrics_path.read_text())
print(json.dumps(metrics, indent=2))
print((OUTPUT_DIR / "training_summary.json").read_text())

In [ ]:
import shutil

adapter_zip = shutil.make_archive("/kaggle/working/medgemma-t1-qlora-balanced", "zip", root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name)
print("Download this adapter ZIP for the one-time test evaluation:", adapter_zip)